# Graph Generation

Third notebook in the split workflow. It loads all-candidate model predictions from `02_models.ipynb` and `02.5_transformer_colab.ipynb`, aggregates mention-level predictions into Knowledge Graph edges, and writes interactive graph HTML files.


## 1. Configuration

In [16]:
from pathlib import Path

# -----------------------------------------------------------------------------
# Shared data folder.
# -----------------------------------------------------------------------------
# All files that are inputs to, outputs from, or shared between multiple models
# are stored in data/. Model-specific artifacts remain in their own folders.
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Input XML.
XML_PATH = DATA_DIR / "bookworm_09062026.xml"

# Shared preprocessing and training files used by the split workflow.
PAGES_CSV = DATA_DIR / "pages.csv"
CHARACTER_GAZETTEER_CSV = DATA_DIR / "character_gazetteer.csv"
CANDIDATE_EXAMPLES_CSV = DATA_DIR / "candidate_examples.csv"
LLM_LABELED_CANDIDATES_CSV = DATA_DIR / "candidate_examples_llm_labeled.csv"
WEAK_LABEL_DISTRIBUTION_CSV = DATA_DIR / "label_distribution.csv"
TRAINING_LABEL_DISTRIBUTION_CSV = DATA_DIR / "training_label_distribution.csv"
TRAIN_CSV = DATA_DIR / "train.csv"
DEV_CSV = DATA_DIR / "dev.csv"
TEST_CSV = DATA_DIR / "test.csv"
SPLIT_LABEL_DISTRIBUTION_CSV = DATA_DIR / "split_label_distribution.csv"
MODEL_COMPARISON_CSV = DATA_DIR / "metrics_model_comparison.csv"
KG_MODEL_COMPARISON_CSV = DATA_DIR / "kg_model_comparison.csv"
RUN_SUMMARY_JSON = DATA_DIR / "model_run_summary.json"
PIPELINE_RUN_METADATA_JSON = DATA_DIR / "pipeline_run_metadata.json"

# Model-specific output folders.
BASELINE_DIR = Path("baseline")  # TF-IDF + Logistic Regression artifacts.
BILSTM_DIR = Path("bilstm")      # BiLSTM artifacts.
TRANSFORMER_DIR = Path("transformer")  # DistilBERT/transformer artifacts from 02.5_transformer_colab.ipynb.
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
BILSTM_DIR.mkdir(parents=True, exist_ok=True)
TRANSFORMER_DIR.mkdir(parents=True, exist_ok=True)

# Relationship labels. "no_relation" is needed as the negative class.
RELATIONSHIPS = [
    "family",
    "romantic",
    "friend_ally",
    "service_retainer",
    "enemy_rival",
    "no_relation",
]

NO_RELATION_LABEL = "no_relation"
POSITIVE_RELATIONS = [label for label in RELATIONSHIPS if label != NO_RELATION_LABEL]

RANDOM_SEED = 42
TEST_SIZE = 0.15
DEV_SIZE = 0.15
MAX_NEGATIVE_RATIO = 2.0
MIN_CONTEXT_CHARS = 25
EDGE_CONFIDENCE_THRESHOLD = 0.95

print(f"XML path: {XML_PATH.resolve()}")
print(f"Shared data folder: {DATA_DIR.resolve()}")
print(f"Baseline output folder: {BASELINE_DIR.resolve()}")
print(f"BiLSTM output folder: {BILSTM_DIR.resolve()}")
print(f"Transformer output folder: {TRANSFORMER_DIR.resolve()}")
print(f"Relationship labels: {RELATIONSHIPS}")


XML path: E:\Natural Language Processing\Project 2\data\bookworm_09062026.xml
Shared data folder: E:\Natural Language Processing\Project 2\data
Baseline output folder: E:\Natural Language Processing\Project 2\baseline
BiLSTM output folder: E:\Natural Language Processing\Project 2\bilstm
Transformer output folder: E:\Natural Language Processing\Project 2\transformer
Relationship labels: ['family', 'romantic', 'friend_ally', 'service_retainer', 'enemy_rival', 'no_relation']


## 2. Imports

In [17]:
import html
import json
import math
import re
import warnings
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 140)
np.random.seed(RANDOM_SEED)

## Load model predictions from 02_models.ipynb and 02.5_transformer_colab.ipynb

Run the models notebooks first. This cell restores the baseline and, when available, BiLSTM and transformer all-candidate predictions used for graph aggregation.


In [18]:
def require_file(path: Path, upstream_notebook: str) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run {upstream_notebook} first.")
    return path

baseline_predictions_path = require_file(BASELINE_DIR / "predictions_all_best_baseline.csv", "02_models.ipynb")
all_pred = pd.read_csv(baseline_predictions_path)

baseline_metrics_path = BASELINE_DIR / "metrics_baseline_variants.csv"
if baseline_metrics_path.exists():
    metrics_df = pd.read_csv(baseline_metrics_path)
    best_variant = str(metrics_df.iloc[0]["variant"]) if len(metrics_df) else "best_baseline"
else:
    metrics_df = pd.DataFrame()
    best_variant = "best_baseline"

bilstm_predictions_path = BILSTM_DIR / "predictions_all_bilstm.csv"
if bilstm_predictions_path.exists():
    bilstm_all_pred = pd.read_csv(bilstm_predictions_path)
    print(f"Loaded BiLSTM predictions: {len(bilstm_all_pred):,} rows")
else:
    print("BiLSTM all-candidate predictions were not found. The BiLSTM graph cells will be skipped until 02_models.ipynb writes them.")


transformer_predictions_path = TRANSFORMER_DIR / "predictions_all_transformer.csv"
if transformer_predictions_path.exists():
    transformer_all_pred = pd.read_csv(transformer_predictions_path)
    print(f"Loaded transformer predictions: {len(transformer_all_pred):,} rows")
else:
    print("Transformer all-candidate predictions were not found. Run 02.5_transformer_colab.ipynb and copy/keep the transformer/ folder before running transformer graph cells.")

print(f"Loaded baseline predictions: {len(all_pred):,} rows from {baseline_predictions_path}")
print(f"Best baseline variant: {best_variant}")


Loaded BiLSTM predictions: 6,031 rows
Loaded transformer predictions: 6,031 rows
Loaded baseline predictions: 6,031 rows from baseline\predictions_all_best_baseline.csv
Best baseline variant: tfidf_logreg_basic


## 13. Aggregate predictions into Knowledge Graph edges

This cell converts mention-level predictions into graph edges:

```text
head_character -- predicted_relationship --> tail_character
```

It removes `no_relation`, applies a confidence threshold, groups repeated evidence for the same pair/relation, and keeps the strongest relation for each pair.

In [19]:
# Relations such as family, romantic, friendship, and rivalry are treated as undirected:
# A --family--> B and B --family--> A are merged into one canonical edge.
# Service/retainer relations are treated as directed because direction matters:
# A --service_retainer--> B is not equivalent to B --service_retainer--> A.
UNDIRECTED_RELATIONS = {
    "family",
    "romantic",
    "friend_ally",
    "enemy_rival",
} & set(RELATIONSHIPS)

DIRECTED_RELATIONS = {
    "service_retainer",
} & set(RELATIONSHIPS)

RELATIONS_WITH_DIRECTION_RULES = UNDIRECTED_RELATIONS | DIRECTED_RELATIONS
UNSPECIFIED_RELATIONS = set(POSITIVE_RELATIONS) - RELATIONS_WITH_DIRECTION_RULES

if UNSPECIFIED_RELATIONS:
    print(
        "Warning: These positive relations are not listed in UNDIRECTED_RELATIONS or DIRECTED_RELATIONS "
        "and will be treated as directed:",
        sorted(UNSPECIFIED_RELATIONS),
    )

print(f"Undirected relations: {sorted(UNDIRECTED_RELATIONS)}")
print(f"Directed relations: {sorted(DIRECTED_RELATIONS)}")


def canonicalize_edge(row: pd.Series) -> pd.Series:
    """Create canonical edge keys.

    For undirected relations, sorted(head, tail) is used so mirrored predictions
    collapse into one edge. For directed relations, the original head/tail order is kept.
    """
    head = row["head"]
    tail = row["tail"]
    relation = row["predicted_label"]

    if relation in UNDIRECTED_RELATIONS:
        edge_head, edge_tail = sorted([head, tail])
        edge_direction = "undirected"
    else:
        # Directed relations and any unspecified positive relation keep model direction.
        edge_head, edge_tail = head, tail
        edge_direction = "directed"

    return pd.Series(
        {
            "edge_head": edge_head,
            "edge_tail": edge_tail,
            "edge_direction": edge_direction,
        }
    )


def aggregate_kg_edges(predictions: pd.DataFrame, confidence_threshold: float = EDGE_CONFIDENCE_THRESHOLD) -> pd.DataFrame:
    """Aggregate mention-level predictions into KG edges.

    Mirrored edges are merged for labels in UNDIRECTED_RELATIONS.
    Directed labels keep their original head -> tail direction.
    """
    edges = predictions.copy()
    edges = edges[edges["predicted_label"] != NO_RELATION_LABEL].copy()
    edges = edges[edges["confidence"].fillna(1.0) >= confidence_threshold].copy()

    empty_columns = [
        "head",
        "tail",
        "relation",
        "edge_direction",
        "evidence_count",
        "mean_confidence",
        "max_confidence",
        "edge_score",
        "evidence",
    ]

    if len(edges) == 0:
        return pd.DataFrame(columns=empty_columns)

    edge_keys = edges.apply(canonicalize_edge, axis=1)
    edges = pd.concat([edges, edge_keys], axis=1)

    grouped_rows = []
    group_cols = ["edge_head", "edge_tail", "predicted_label", "edge_direction"]

    for (edge_head, edge_tail, relation, edge_direction), group in edges.groupby(group_cols):
        evidence_count = len(group)
        mean_conf = float(group["confidence"].mean())
        max_conf = float(group["confidence"].max())
        edge_score = mean_conf * math.log1p(evidence_count)

        evidence = " | ".join(
            group
            .sort_values("confidence", ascending=False)["context"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(3)
            .tolist()
        )

        grouped_rows.append(
            {
                "head": edge_head,
                "tail": edge_tail,
                "relation": relation,
                "edge_direction": edge_direction,
                "evidence_count": evidence_count,
                "mean_confidence": mean_conf,
                "max_confidence": max_conf,
                "edge_score": edge_score,
                "evidence": evidence,
            }
        )

    kg_edges = pd.DataFrame(grouped_rows)

    # Keep only the highest-scoring relation per canonical character pair.
    # For undirected relations, this removes mirrored duplicates such as A-B and B-A.
    # For directed relations, the original direction is preserved.
    kg_edges = (
        kg_edges.sort_values(["head", "tail", "edge_score"], ascending=[True, True, False])
        .drop_duplicates(subset=["head", "tail"], keep="first")
        .sort_values("edge_score", ascending=False)
        .reset_index(drop=True)
    )

    return kg_edges


kg_edges_df = aggregate_kg_edges(all_pred)
kg_edges_path = BASELINE_DIR / "kg_edges_best_baseline.csv"
kg_edges_df.to_csv(kg_edges_path, index=False)

print(f"KG edges saved to: {kg_edges_path}")
print(f"Predicted nodes: {len(set(kg_edges_df['head']).union(set(kg_edges_df['tail'])) if len(kg_edges_df) else set())}")
print(f"Predicted edges: {len(kg_edges_df):,}")

if len(kg_edges_df):
    print("\nEdge direction counts:")
    display(kg_edges_df["edge_direction"].value_counts().rename_axis("edge_direction").reset_index(name="count"))

display(kg_edges_df.head(20))


Undirected relations: ['enemy_rival', 'family', 'friend_ally', 'romantic']
Directed relations: ['service_retainer']
KG edges saved to: baseline\kg_edges_best_baseline.csv
Predicted nodes: 191
Predicted edges: 653

Edge direction counts:


,edge_direction,count
0,undirected,562
1,directed,91


,head,tail,relation,edge_direction,evidence_count,mean_confidence,max_confidence,edge_score,evidence
0,Justus,Rihyarda,family,undirected,5,0.985928,0.997226,1.766545,"Infobox field family/Mother: Rihyarda | Infobox field family/Son: Justus | He bears a strong resemblance to his mother, Rihyarda, and hi..."
1,Gieselfried,Letizia,family,undirected,5,0.980871,0.996931,1.757484,"Infobox field family/Father: Drewanchel Archducal Family Member (Biological) Gieselfried (Adoptive, Deceased) | Infobox field family/Mot..."
2,Rihyarda,Traugott,family,undirected,5,0.979615,0.987158,1.755234,"Traugott is the son of Karstedt's younger brother and Gudrun, Rihyarda's daughter. | Infobox field family/Offspring: Traugott (Grandson)..."
3,Detlinde,Letizia,family,undirected,4,0.991210,0.993799,1.595292,Infobox field family/Sister: Detlinde (Adoptive Half-Sister) Alstede (Adoptive Half-Sister) | Infobox field family/Sister: Alstede Letiz...
4,Elvira,Previous Count Leisegang,family,undirected,4,0.987765,0.993370,1.589747,Infobox field family/Relatives: Previous Count Leisegang (Great-Grandfather) Count Groschel (Cousin) Traugott (Nephew) | Infobox field f...
5,Blasius,Letizia,family,undirected,4,0.985653,0.994674,1.586348,Infobox field family/Uncle: Wolfram (Deceased) Blasius | Infobox field family/Relatives: Aurelia (Cousin) Martina (Cousin) Letizia (Niec...
6,Hartmut,Leberecht,family,undirected,4,0.984416,0.999527,1.584357,"Infobox field family/Father: Leberecht | He is the father of three sons, of whom Hartmut is the youngest and Oliswalt the oldest. | Info..."
7,Eglantine,Hildebrand,family,undirected,4,0.984298,0.995107,1.584167,Infobox field family/Relatives: Anastasius (Cousin) Sigiswald (Cousin) Hildebrand (Cousin) Nahelache's Son (Nephew) | Infobox field fami...
8,Gudrun,Traugott,family,undirected,4,0.984016,0.998269,1.583712,"Infobox field family/Mother: Gudrun | Infobox field family/Son: Traugott | Traugott is the son of Karstedt's younger brother and Gudrun,..."
9,Eglantine,Sigiswald,family,undirected,4,0.981926,0.995107,1.580350,Infobox field family/Relatives: Anastasius (Cousin) Sigiswald (Cousin) Hildebrand (Cousin) Nahelache's Son (Nephew) | Infobox field fami...


## 14. Aggregate BiLSTM predictions into Knowledge Graph edges

This cell applies the same KG aggregation function to the BiLSTM predictions so that the baseline and neural model are compared using the same graph-construction logic.


In [20]:
if "bilstm_all_pred" in globals():
    kg_edges_bilstm_df = aggregate_kg_edges(bilstm_all_pred)
    kg_edges_bilstm_path = BILSTM_DIR / "kg_edges_bilstm.csv"
    kg_edges_bilstm_df.to_csv(kg_edges_bilstm_path, index=False)

    print(f"BiLSTM KG edges saved to: {kg_edges_bilstm_path}")
    print(
        "BiLSTM predicted nodes:",
        len(set(kg_edges_bilstm_df["head"]).union(set(kg_edges_bilstm_df["tail"]))) if len(kg_edges_bilstm_df) else 0,
    )
    print(f"BiLSTM predicted edges: {len(kg_edges_bilstm_df):,}")

    if len(kg_edges_bilstm_df):
        print("\nBiLSTM edge direction counts:")
        display(kg_edges_bilstm_df["edge_direction"].value_counts().rename_axis("edge_direction").reset_index(name="count"))

    display(kg_edges_bilstm_df.head(20))
else:
    print("BiLSTM predictions were not found. Run Section 11 before this cell.")


BiLSTM KG edges saved to: bilstm\kg_edges_bilstm.csv
BiLSTM predicted nodes: 236
BiLSTM predicted edges: 976

BiLSTM edge direction counts:


,edge_direction,count
0,undirected,739
1,directed,237


,head,tail,relation,edge_direction,evidence_count,mean_confidence,max_confidence,edge_score,evidence
0,Adolphine,Sigiswald,romantic,undirected,12,0.978033,0.995014,2.508604,"Infobox field family/Spouse: Nahelache (First Wife, Former Second Wife) Adolphine (First Wife, Former) Rozemyne (Engaged, Cancelled) | W..."
1,Justus,Rihyarda,family,undirected,9,0.979309,0.994610,2.254943,"He bears a strong resemblance to his mother, Rihyarda, and his sister, Gudrun. | At some point, Rihyarda started a family and had two ch..."
2,Lieseleta,Thorsten,romantic,undirected,7,0.979583,0.994183,2.036986,"Infobox field family/Spouse: Lieseleta (Engaged, Cancelled) | Infobox field family/Spouse: Unnamed Man (Engaged) Thorsten (Engaged, Canc..."
3,Georgine,Veronica,family,undirected,7,0.972153,0.991785,2.021536,"She is the mother of Georgine, Constanze and Sylvester. | When her parents made it clear to Georgine, that she would never become the Au..."
4,Magdalena,Raublut,enemy_rival,undirected,6,0.987125,0.997467,1.920856,"In the confrontation with their enemy, Magdalena and Werdekraf attack Raublut together. | This makes Raublut consider her the lesser thr..."
5,Oswald,Wilfried,service_retainer,directed,6,0.986357,0.997314,1.919363,"Oswald (オズヴァルト, Ozuvaruto) is an archnoble of Ehrenfest and formerly Wilfried's head attendant. | Two days after returning with his lord..."
6,Gieselfried,Letizia,family,undirected,6,0.981585,0.993236,1.910077,"Infobox field family/Father: Drewanchel Archducal Family Member (Biological) Gieselfried (Adoptive, Deceased) | Infobox field family/Mot..."
7,Martina,Detlinde,service_retainer,directed,6,0.981266,0.999440,1.909455,"Martina (マルティナ, Marutina) was an apprentice Ahrensbach archattendant serving Detlinde. | Infobox field occupation: Prisoner Apprentice A..."
8,Adolphine,Ortwin,family,undirected,6,0.976439,0.985561,1.900062,"Adolphine grew up closest to her younger brother Ortwin, since they shared the same mother. | Infobox field family/Sister: Adolphine | H..."
9,Konrad,Philine,family,undirected,6,0.973549,0.988004,1.894438,"Konrad was born in the city of Ehrenfest as the laynoble son of Kashick and his wife Theresia, and is the younger brother of Philine. | ..."


## 15. Aggregate transformer predictions into Knowledge Graph edges


In [21]:
if "transformer_all_pred" in globals():
    kg_edges_transformer_df = aggregate_kg_edges(transformer_all_pred)
    kg_edges_transformer_path = TRANSFORMER_DIR / "kg_edges_transformer.csv"
    kg_edges_transformer_df.to_csv(kg_edges_transformer_path, index=False)

    print(f"Transformer KG edges saved to: {kg_edges_transformer_path}")
    print(
        "Transformer predicted nodes:",
        len(set(kg_edges_transformer_df["head"]).union(set(kg_edges_transformer_df["tail"]))) if len(kg_edges_transformer_df) else 0,
    )
    print(f"Transformer predicted edges: {len(kg_edges_transformer_df):,}")

    if len(kg_edges_transformer_df):
        print("\nTransformer edge direction counts:")
        display(kg_edges_transformer_df["edge_direction"].value_counts().rename_axis("edge_direction").reset_index(name="count"))

    display(kg_edges_transformer_df.head(20))
else:
    print("Transformer predictions were not found. Run 02.5_transformer_colab.ipynb before this cell.")


Transformer KG edges saved to: transformer\kg_edges_transformer.csv
Transformer predicted nodes: 251
Transformer predicted edges: 1,297

Transformer edge direction counts:


,edge_direction,count
0,undirected,1042
1,directed,255


,head,tail,relation,edge_direction,evidence_count,mean_confidence,max_confidence,edge_score,evidence
0,Adolphine,Sigiswald,romantic,undirected,19,0.978115,0.986340,2.930170,"After graduating from the Royal Academy, she married Prince Sigiswald as his first wife. | With Eglantine now removed from his considera..."
1,Lieseleta,Thorsten,romantic,undirected,12,0.981133,0.984913,2.516556,"Infobox field family/Spouse: Unnamed Man (Engaged) Thorsten (Engaged, Cancelled) Uderick (Engaged, Cancelled) | With her being without a..."
2,Effa,Gunther,romantic,undirected,11,0.977867,0.985552,2.429907,"Thanks to Gunther's passion, his marriage proposal was consented, if Effa wanted to marry him. | A good-natured and hard-working woman, ..."
3,Georgine,Veronica,family,undirected,10,0.982760,0.989512,2.356555,"Infobox field family/Mother: Veronica | She bears a striking resemblance to her mother, Veronica . | She is the mother of Georgine, Cons..."
4,Hannelore,Lestilaut,family,undirected,9,0.986282,0.988420,2.270998,*Kenntrips - Kenntrips is an apprentice archscholar and cousin of Lestilaut and Hannelore. | *Rasantark - A male apprentice archknight a...
5,Brigitte,Hassheit,romantic,undirected,9,0.978013,0.985417,2.251957,"Hassheit (ハスハイト, Hasuhaito) is a mednoble of Ehrenfest and Brigitte's former fiancé. | Brigitte's became engaged to Hassheit soon after ..."
6,Justus,Rihyarda,family,undirected,8,0.987366,0.989311,2.169464,"Infobox field family/Mother: Rihyarda | His mother is Rihyarda, Rozemyne's former head attendant in the castle. | At some point, Rihyard..."
7,Lieseleta,Uderick,romantic,undirected,8,0.983383,0.985879,2.160714,Uderick (ウーデリック) is an adult medattendant of Ehrenfest who was Lieseleta's previous fiancé prior to Lieseleta reaching archnoble level m...
8,Ralfrieda,Trauerqual,romantic,undirected,8,0.981629,0.986745,2.156858,"Ralfrieda was born in the duchy of Gilessenmeyer and later in life married the future Zent Trauerqual. | However for political reasons, ..."
9,Adelbert,Sylvester,family,undirected,8,0.980616,0.988566,2.154635,"Infobox field family/Father: Adelbert (Deceased) | Sylvester is the son of Adelbert, the sixth Aub Ehrenfest and Veronica. | Adelbert wa..."


## 16. Create PyVis Knowledge Graph visualization

If `pyvis` is installed, this cell creates an interactive HTML graph. If not, it writes a simpler HTML edge table so the pipeline still completes.

In [22]:
def inject_relation_filter(html_path: Path, relations: list[str]) -> None:
    """Inject relation checkbox filters into a PyVis HTML file."""
    html_path = Path(html_path)
    html_text = html_path.read_text(encoding="utf-8")

    checkbox_html = "\n".join(
        f"""
        <label style="display:block; margin: 3px 0;">
            <input type="checkbox" class="relation-filter" value="{relation}" checked>
            {relation}
        </label>
        """
        for relation in relations
    )

    control_panel = f"""
    <div id="relation-filter-panel" style="
        position: fixed;
        top: 10px;
        right: 10px;
        z-index: 9999;
        background: white;
        border: 1px solid #ccc;
        border-radius: 6px;
        padding: 10px 12px;
        font-family: Arial, sans-serif;
        font-size: 13px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.15);
        max-width: 240px;
    ">
        <strong>Filter relations</strong>
        <div style="margin-top: 6px;">
            {checkbox_html}
        </div>
        <button id="select-all-relations" style="margin-top: 8px;">Select all</button>
        <button id="clear-all-relations" style="margin-top: 8px;">Clear all</button>
    </div>
    """

    filter_script = """
    <script type="text/javascript">
    document.addEventListener("DOMContentLoaded", function () {
        if (typeof edges === "undefined" || typeof nodes === "undefined" || typeof network === "undefined") {
            console.warn("PyVis variables not found; relation filter was not attached.");
            return;
        }

        var allEdges = edges.get();
        var allNodes = nodes.get();

        function getEdgeRelation(edge) {
            // Prefer custom relation metadata.
            // Fall back to edge label, because PyVis always keeps the label.
            return edge.relation || edge.label;
        }

        function updateGraphFilter() {
            var checkedRelations = Array.from(
                document.querySelectorAll(".relation-filter:checked")
            ).map(function (box) {
                return box.value;
            });

            var visibleNodeIds = new Set();
            var edgeUpdates = [];

            allEdges.forEach(function (edge) {
                var relation = getEdgeRelation(edge);
                var isVisible = checkedRelations.includes(relation);

                edgeUpdates.push({
                    id: edge.id,
                    hidden: !isVisible
                });

                if (isVisible) {
                    visibleNodeIds.add(edge.from);
                    visibleNodeIds.add(edge.to);
                }
            });

            var nodeUpdates = allNodes.map(function (node) {
                return {
                    id: node.id,
                    hidden: !visibleNodeIds.has(node.id)
                };
            });

            edges.update(edgeUpdates);
            nodes.update(nodeUpdates);

            network.redraw();
        }

        document.querySelectorAll(".relation-filter").forEach(function (box) {
            box.addEventListener("change", updateGraphFilter);
        });

        document.getElementById("select-all-relations").addEventListener("click", function () {
            document.querySelectorAll(".relation-filter").forEach(function (box) {
                box.checked = true;
            });
            updateGraphFilter();
        });

        document.getElementById("clear-all-relations").addEventListener("click", function () {
            document.querySelectorAll(".relation-filter").forEach(function (box) {
                box.checked = false;
            });
            updateGraphFilter();
        });
    });
    </script>
    """

    html_text = html_text.replace("<body>", f"<body>\n{control_panel}")
    html_text = html_text.replace("</body>", f"{filter_script}\n</body>")

    html_path.write_text(html_text, encoding="utf-8")

def write_pyvis_graph(edges_df: pd.DataFrame, output_path: Path, title: str = "TF-IDF + Logistic Regression KG"):
    """Write an interactive PyVis graph if pyvis is available; otherwise write a fallback HTML table."""
    output_path = Path(output_path)

    if len(edges_df) == 0:
        output_path.write_text("<html><body><h1>No edges passed the threshold.</h1></body></html>", encoding="utf-8")
        return "empty"

    try:
        from pyvis.network import Network

        net = Network(height="800px", width="100%", directed=True, notebook=False)
        net.barnes_hut()

        net.set_options("""
        {
          "physics": {
            "enabled": true,
            "barnesHut": {
              "gravitationalConstant": -25000,
              "centralGravity": 1,
              "springLength": 150,
              "springConstant": 0.04,
              "damping": 0.18,
              "avoidOverlap": 1
            },
            "stabilization": {
              "enabled": true,
              "iterations": 1200,
              "updateInterval": 25
            }
          },
          "nodes": {
            "font": {
              "size": 30
            }
          },
          "edges": {
            "font": {
              "size": 8
            },
            "smooth": {
              "enabled": true,
              "type": "dynamic"
            }
          },
          "interaction": {
            "dragNodes": true,
            "dragView": true,
            "zoomView": true
          }
        }
        """)

        degree_counter = Counter(edges_df["head"]) + Counter(edges_df["tail"])
        nodes = sorted(set(edges_df["head"]).union(set(edges_df["tail"])))

        for node in nodes:
            degree = degree_counter[node]
            net.add_node(
                node,
                label=node,
                size=min(35, 10 + 2 * degree),
                title=f"Character: {node}<br>Degree: {degree}",
            )

        for edge_idx, (_, row) in enumerate(edges_df.iterrows()):
            edge_direction = row.get("edge_direction", "directed")
            tooltip = (
                f"Relation: {row['relation']}<br>"
                f"Direction: {edge_direction}<br>"
                f"Mean confidence: {row['mean_confidence']:.3f}<br>"
                f"Evidence count: {int(row['evidence_count'])}<br>"
                f"Evidence: {row['evidence']}"
            )

            # Keep arrows only for directed relations such as service_retainer.
            # Undirected relations such as family, romantic, friend_ally, and enemy_rival
            # are displayed without arrowheads.
            if edge_direction == "undirected":
                arrows = {
                    "to": {"enabled": False},
                    "from": {"enabled": False},
                    "middle": {"enabled": False},
                }
            else:
                arrows = {
                    "to": {"enabled": True},
                    "from": {"enabled": False},
                    "middle": {"enabled": False},
                }

            net.add_edge(
                row["head"],
                row["tail"],
                label=row["relation"],
                width=0.5,
                arrows=arrows,
                title=tooltip,
                id=f"edge_{edge_idx}",
                relation=row["relation"],
                edge_direction=edge_direction,
            )

        net.write_html(str(output_path))
        relations = sorted(edges_df["relation"].dropna().unique().tolist())
        inject_relation_filter(output_path, relations)
        return "pyvis"

    except Exception as exc:
        fallback_html = f"""
        <html>
        <head><meta charset=\"utf-8\"><title>{title}</title></head>
        <body>
        <h1>{title}</h1>
        <p>PyVis graph could not be created: {html.escape(str(exc))}</p>
        {edges_df.to_html(index=False, escape=True)}
        </body>
        </html>
        """
        output_path.write_text(fallback_html, encoding="utf-8")
        return "fallback_html"


graph_path = BASELINE_DIR / "kg_tfidf_logreg_best_baseline.html"
graph_status = write_pyvis_graph(
    kg_edges_df,
    graph_path,
    title=f"Baseline KG: {best_variant}",
)

print(f"Graph status: {graph_status}")
print(f"Graph written to: {graph_path}")

if "kg_edges_bilstm_df" in globals():
    graph_path_bilstm = BILSTM_DIR / "kg_bilstm.html"
    graph_status_bilstm = write_pyvis_graph(
        kg_edges_bilstm_df,
        graph_path_bilstm,
        title="BiLSTM KG",
    )
    print(f"BiLSTM graph status: {graph_status_bilstm}")
    print(f"BiLSTM graph written to: {graph_path_bilstm}")
else:
    print("BiLSTM KG edges were not found. Run Section 14 before creating the BiLSTM graph.")

if "kg_edges_transformer_df" in globals():
    graph_path_transformer = TRANSFORMER_DIR / "kg_transformer.html"
    graph_status_transformer = write_pyvis_graph(
        kg_edges_transformer_df,
        graph_path_transformer,
        title="Transformer KG",
    )
    print(f"Transformer graph status: {graph_status_transformer}")
    print(f"Transformer graph written to: {graph_path_transformer}")
else:
    print("Transformer KG edges were not found. Run the transformer KG aggregation cell before creating the transformer graph.")


Graph status: pyvis
Graph written to: baseline\kg_tfidf_logreg_best_baseline.html
BiLSTM graph status: pyvis
BiLSTM graph written to: bilstm\kg_bilstm.html
Transformer graph status: pyvis
Transformer graph written to: transformer\kg_transformer.html
